#Spider A Large-Scale Human-Labeled Dataset (Text-to-SQL Task)

Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task

Authors: Tao Yu, Rui Zhang, Kai Yang, Michihiro Yasunaga, Dongxu Wang, Zifan Li, James Ma, Irene Li, Qingning Yao, Shanelle Roman, Zilin Zhang, Dragomir Radev

"We present Spider, a large-scale, complex and cross-domain semantic parsing and text-to-SQL dataset annotated by 11 college students. It consists of 10,181 questions and 5,693 unique complex SQL queries on 200 databases with multiple tables, covering 138 different domains. We define a new complex and cross-domain semantic parsing and text-to-SQL task where different complex SQL queries and databases appear in train and test sets. In this way, the task requires the model to generalize well to both new SQL queries and new database schemas. Spider is distinct from most of the previous semantic parsing tasks because they all use a single database and the exact same programs in the train set and the test set. We experiment with various state-of-the-art models and the best model achieves only 12.4% exact matching accuracy on a database split setting. This shows that Spider presents a strong challenge for future research. "

https://arxiv.org/abs/1809.08887

In [6]:
# Import Dataset 
import pandas as pd

data = pd.read_csv("/kaggle/input/understanding-contextual-questions-answers/train.csv", nrows =1000)
data.tail()

,context,question,answer
995,"CREATE TABLE player (Sponsor_name VARCHAR, Res...",Show the names of sponsors of players whose re...,SELECT Sponsor_name FROM player WHERE Residenc...
996,"CREATE TABLE player (Player_name VARCHAR, Vote...",What is the name of the player with the larges...,SELECT Player_name FROM player ORDER BY Votes ...
997,CREATE TABLE player (Occupation VARCHAR),Show different occupations along with the numb...,"SELECT Occupation, COUNT(*) FROM player GROUP ..."
998,CREATE TABLE player (Occupation VARCHAR),Please show the most common occupation of play...,SELECT Occupation FROM player GROUP BY Occupat...
999,CREATE TABLE player (Residence VARCHAR),Show the residences that have at least two pla...,SELECT Residence FROM player GROUP BY Residenc...


In [2]:
import re

# Install Keras 3 last. 
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

import os

os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

import keras
import keras_nlp

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflowjs 4.16.0 requires packaging~=23.1, but you have packaging 21.3 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.0 requires keras<2.16,>=2.15.0, but you have keras 3.8.0 which is incompatible.
tensorflowjs 4.16.0 requires packaging~=23.1, but you have packaging 21.3 which is incompatible.


2025-02-16 07:33:13.732270: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-16 07:33:13.732330: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-16 07:33:13.734025: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Gemma

In [3]:
#Create the model using the from_preset method
#gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("/kaggle/input/gemma/keras/gemma_2b_en/2")
#Create the model using the from_preset method
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("/kaggle/input/codegemma/keras/code_gemma_1.1_2b_en/2")

normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


In [4]:
#Example queries
gemma_lm.generate("What is the name of the player with the largest number of votes?", max_length=64)

'What is the name of the player with the largest number of votes? (1).ipynb\n<|fim_prefix|><|fim_suffix|>\n# %%\n#The player with the largest number of votes is "Michael"\n<|fim_middle|># %%\nimport pandas as pd\n# %%\ndf = pd.read_csv(\'votes.csv'

In [7]:
#Prepare the dataset for fine-tuning
dataset = []
    
for index, row in data.iterrows():
    question, answer = row['question'], row['answer']
    template = (f"Question:\n{question}\n\nAnswer:\n{answer}")
    dataset.append(template)

# Pretrain

In [9]:
print(gemma_lm.generate("SQL:How many heads of the departments are older than 56 ?\n", max_length=256))

SQL:How many heads of the departments are older than 56 ?
SELECT d.dept_name,
       e.birth_date,
       e.first_name,
       e.last_name,
       e.gender,
       e.hire_date,
       COUNT(e.emp_no) OVER (PARTITION BY d.dept_name) AS total_employees
FROM employees e
JOIN dept_emp de ON e.emp_no = de.emp_no
JOIN departments d ON de.dept_no = d.dept_no
WHERE e.birth_date > '1956-01-01';
<|file_separator|>


# Enable LoRA

In [ ]:
#Enable LoRA for the model and set the LoRA rank to 64.
gemma_lm.backbone.enable_lora(rank=64)

# Memory Control - Epochs

In [ ]:
# Limit the input sequence length to 512 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 512
# Use AdamW 
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()])

gemma_lm.fit(dataset, epochs=1, batch_size=1)#Original was Epochs=15

In [ ]:
print(gemma_lm.generate("How many heads of the departments are older than 56 ?", max_length=256))

In [ ]:
print(gemma_lm.generate("What are the maximum and minimum budget of the departments?", max_length=256))

In [ ]:
print(gemma_lm.generate("What are the names of the heads who are born outside the California state?", max_length=256))

In [ ]:
print(gemma_lm.generate("Show the names of sponsors of players whose residence is either Brandon or Birtle", max_length=256))

In [ ]:
print(gemma_lm.generate("List the total number of horses on farms in ascending order.", max_length=256))

In [ ]:
print(gemma_lm.generate("Please show the most common occupation of players.", max_length=256))

In [ ]:
print(gemma_lm.generate("What are the hosts of competitions whose theme is not Aliens?", max_length=256))